# Exportación del modelo: TensorFlow.js y TFLite

El mismo modelo entrenado sale en dos formatos:

- **TensorFlow.js** para el navegador, que es donde corre la app hoy (y en el
  celular, abriéndola desde el navegador);
- **TFLite** para una futura app nativa con `react-native-fast-tflite`.

La condición es que **den el mismo resultado**. La última celda lo comprueba
sobre datos reales en vez de asumirlo: exportar no es una operación neutra y
la cuantización puede cambiar predicciones.

In [ ]:
!pip install -q tensorflowjs

import json, os
import numpy as np
import tensorflow as tf
from google.colab import drive

drive.mount('/content/drive')
BASE = '/content/drive/MyDrive/traduce_senas'
MODELO = f'{BASE}/modelos/modelo_unificado.keras'
SALIDA = f'{BASE}/export'
os.makedirs(SALIDA, exist_ok=True)

modelo = tf.keras.models.load_model(MODELO)
glosas = json.load(open(f'{BASE}/modelos/glosas.json', encoding='utf-8'))
print(modelo.input_shape, '->', modelo.output_shape, '|', len(glosas), 'glosas')

## 1. TensorFlow.js

**Sin cuantizar.** Comprimir a 8 bits ahorraría unos 200 KB, pero cambia las
salidas y rompe la paridad con TFLite, que es justo lo que hay que garantizar.

In [ ]:
import tensorflowjs as tfjs

TFJS_DIR = f'{SALIDA}/tfjs'
tfjs.converters.save_keras_model(modelo, TFJS_DIR)

# La app espera las glosas junto al modelo, en /models/lsc/.
with open(f'{TFJS_DIR}/glosas.json', 'w', encoding='utf-8') as fh:
    json.dump(glosas, fh, ensure_ascii=False, indent=2)

!ls -lh {TFJS_DIR}

## 2. TFLite

In [ ]:
conv = tf.lite.TFLiteConverter.from_keras_model(modelo)
# Las GRU necesitan los operadores de TF ademas de los de TFLite.
conv.target_spec.supported_ops = [
    tf.lite.OpsSet.TFLITE_BUILTINS,
    tf.lite.OpsSet.SELECT_TF_OPS,
]
conv._experimental_lower_tensor_list_ops = False

tflite = conv.convert()
RUTA_TFLITE = f'{SALIDA}/modelo.tflite'
open(RUTA_TFLITE, 'wb').write(tflite)
print(f'{len(tflite)/1e6:.2f} MB -> {RUTA_TFLITE}')

## 3. ¿Dan lo mismo?

Se pasan las secuencias de prueba por el modelo original y por el `.tflite`.
El criterio: **misma clase en el 100 % de los casos** y diferencia de
puntajes por debajo de 1e-4.

In [ ]:
datos = np.load(f'{BASE}/datasets/dataset.npz', allow_pickle=True)
X = datos['X_test'][:100].astype(np.float32)
if len(X) == 0:
    X = datos['X_train'][:100].astype(np.float32)
print('secuencias de prueba:', len(X))

keras_out = modelo.predict(X, verbose=0)

interp = tf.lite.Interpreter(model_content=tflite)
interp.allocate_tensors()
ent, sal = interp.get_input_details()[0], interp.get_output_details()[0]

tflite_out = []
for muestra in X:
    interp.set_tensor(ent['index'], muestra[None, ...])
    interp.invoke()
    tflite_out.append(interp.get_tensor(sal['index'])[0])
tflite_out = np.array(tflite_out)

iguales = (keras_out.argmax(1) == tflite_out.argmax(1)).mean()
dif = float(np.abs(keras_out - tflite_out).max())
print(f'misma clase: {iguales:.1%}')
print(f'mayor diferencia de puntaje: {dif:.2e}')
print('PARIDAD OK' if iguales == 1.0 and dif < 1e-4 else 'REVISAR: los dos formatos no coinciden')

## 4. Qué hacer con los archivos

Descarga la carpeta `export/tfjs` de tu Drive y cópiala en el repo del
frontend como:

```
frontend/src/web/models/lsc/
    model.json
    group1-shard1of1.bin
    glosas.json
```

La app busca ahí sola (`WORD_MODEL_URL` en `app/config/api.config.ts`). Si el
archivo no está, sigue funcionando con los 7 gestos de Google, así que no se
rompe nada mientras tanto.

El `.tflite` queda guardado para la app nativa; no se usa en la versión web.

In [ ]:
!ls -lhR {SALIDA}
print('\nTamaño total del modelo web:',
      f"{sum(os.path.getsize(os.path.join(TFJS_DIR, f)) for f in os.listdir(TFJS_DIR))/1e6:.2f} MB")